这里说频率是指一个机场航班出现的频率

## 加载数据

数据已经预处理过了

In [12]:
import pandas as pd

# 加载数据
data = pd.read_csv('../../data-hh/my/hh_result/result_all.csv', dtype={'aircraft': str})

# 查看前几行数据，确保加载成功
print(data.head())

print(data.info())

         flt_no bd_type    cap aircraft  legs  leg_no  duration  pax  \
0  KgJrsp7Jd78=      窄体  132.0      319     1       1      1.07   25   
1  P9IRwar34h0=      窄体  189.0      321     1       1      1.38  151   
2  mJitm0UDfM4=      窄体  132.0      319     1       1      1.57   38   
3  jXr97M1wpn4=      窄体  132.0      319     1       1      1.58  109   
4  izjfHOxAho4=      窄体  132.0      319     1       1      1.80  124   

              a             b  ...  year  month  day  weekday  hour  minute  \
0  KNqX4/Q5Noc=  HexFWXqbb8I=  ...  2023     10    1        6    12      15   
1  AKQNtuL5r6Q=  Mv6HkAiSLUk=  ...  2023     10    1        6    20      10   
2  n465JzB8Rrw=  N4hmDZN/CJQ=  ...  2023     10    1        6    16      45   
3  N4hmDZN/CJQ=  n465JzB8Rrw=  ...  2023     10    1        6    19      40   
4  5t+HPO9Mu/w=  X5e5r3CS4OA=  ...  2023     10    1        6    13       5   

   second          from            to   unit_price  
0       0  KNqX4/Q5Noc=  HexFWXqbb8I=  

## 编码分类变量

### 新增城市标签

In [14]:
# 加载字典
with open('../../data-hh/my/encoder/city_labels_航班频率加权图标签.json', 'r') as file:
    city_labels_loaded = json.load(file)

print("加载的字典：", city_labels_loaded)

加载的字典： {'+3eoTOkHfoU=': 2, '0Koyk9pJc0s=': 2, '4j3sFPu6XuA=': 2, '9obCP1myMDU=': 0, 'CnmNtb5MOJA=': 0, 'CzI09pVjQzQ=': 2, 'DInJHXgQayE=': 2, 'EIsXRbFUH34=': 2, 'MSFyivV4ETI=': 2, 'N4hmDZN/CJQ=': 0, 'UO2RniR7lJ8=': 2, 'XUXDuFGVfBM=': 2, 'ZCvDTbZuS8Q=': 2, 'gK3uAaRHrOs=': 0, 'jVYRvyHjuvM=': 2, '+O1iBQtlFGU=': 0, '/M8gHzjxpNU=': 0, '0J4jz5aCatU=': 0, 'AKQNtuL5r6Q=': 0, 'AlGb30xUGAU=': 0, 'C3VfOpfT580=': 0, 'Eg2bemFsI9I=': 0, 'FWD6O2fHsIU=': 0, 'KETr2NAmAHE=': 0, 'KFsTKxkC/6g=': 0, 'Qe7TEMbQt6o=': 0, 'QyEgIt0M8Sg=': 0, 'So/c8CkA/Xs=': 0, 'XuLAnICDnIM=': 0, 'a2cXIUGpIbw=': 0, 'g1hKMyP1V8I=': 0, 'iX2Pe2pxiqQ=': 1, 'lI1bSVMW3Tg=': 0, 'mljW2xeLSiI=': 1, 'ngSgjHYUqxE=': 0, 'tKjndGSl9NQ=': 0, 'vbo2dwBOeps=': 0, 'yypkiQCX5lk=': 0, '+S5aV4Y0s8w=': 0, 'Szk6NJM5C9Y=': 0, 'hwin1ipuzwk=': 0, 'nJFTk7JNmpw=': 0, 'wSF4qXvI044=': 0, 'xIqe+QQIQXQ=': 0, 'zwVGiYamVc0=': 0, '+cjxQ01BFmw=': 1, '/qxKZMQeBus=': 0, '07fT2dsZOJM=': 0, '1/9nd1jNCE0=': 0, '2WRfpdopfDw=': 0, '3nMB4t9SNGw=': 0, '4J2zqjPuTyI=': 0, '4Li

In [15]:
# 使用 map 对 'a', 'b', 'c', 'from', 'to' 列进行标签化，新增对应的标签列
data['a_label'] = data['a'].map(city_labels_loaded)
data['b_label'] = data['b'].map(city_labels_loaded)
data['c_label'] = data['c'].map(city_labels_loaded)
data['from_label'] = data['from'].map(city_labels_loaded)
data['to_label'] = data['to'].map(city_labels_loaded)

### 统计不同城市的频率

In [16]:
# 将 from 和 to 两列合并，并统计每个城市的出现次数
city_count_series = pd.concat([data['from'], data['to']]).value_counts()

# 转换为 city_count 数据框
city_count = city_count_series.reset_index()
city_count.columns = ['city', 'count']

# 3. 创建 city_map，将每个城市映射为其出现次数（或者任何你需要的值）
city_map = dict(zip(city_count['city'], city_count['count']))

# 输出结果
print(city_count)

             city   count
0    vbo2dwBOeps=  478332
1    /M8gHzjxpNU=  450316
2    gK3uAaRHrOs=  446006
3    So/c8CkA/Xs=  418933
4    0J4jz5aCatU=  408305
..            ...     ...
247  iPk6Y04ZgnI=     122
248  MgrS1LT2KO4=      91
249  DwCmftDLjPw=      62
250  wbf1sxPhwNM=      60
251  Eed3EdkMDqg=      49

[252 rows x 2 columns]


使用 json 保存和加载 city_map

In [17]:
import json

# 保存 city_map 到 JSON 文件
with open('../../data-hh/my/encoder/city_map_频率编码.json', 'w') as f:
    json.dump(city_map, f)

print("city_map_频率编码 已保存到 city_map_频率编码")

city_map_频率编码 已保存到 city_map_频率编码


In [18]:
# 使用 city_map 替换指定列的值
columns_to_replace = ['a', 'b', 'c', 'from', 'to']

# 遍历指定列并直接用 map 映射
for col in columns_to_replace:
    data[col] = data[col].map(city_map)

# 输出结果
print(city_count)
print(data)

             city   count
0    vbo2dwBOeps=  478332
1    /M8gHzjxpNU=  450316
2    gK3uAaRHrOs=  446006
3    So/c8CkA/Xs=  418933
4    0J4jz5aCatU=  408305
..            ...     ...
247  iPk6Y04ZgnI=     122
248  MgrS1LT2KO4=      91
249  DwCmftDLjPw=      62
250  wbf1sxPhwNM=      60
251  Eed3EdkMDqg=      49

[252 rows x 2 columns]
               flt_no bd_type    cap aircraft  legs  leg_no  duration  pax  \
0        KgJrsp7Jd78=      窄体  132.0      319     1       1      1.07   25   
1        P9IRwar34h0=      窄体  189.0      321     1       1      1.38  151   
2        mJitm0UDfM4=      窄体  132.0      319     1       1      1.57   38   
3        jXr97M1wpn4=      窄体  132.0      319     1       1      1.58  109   
4        izjfHOxAho4=      窄体  132.0      319     1       1      1.80  124   
...               ...     ...    ...      ...   ...     ...       ...  ...   
5999220  BzUm4im0EqA=      窄体  152.0      320     1       1      3.03   99   
5999221  w+GXbx7u3EM=      窄体  152.0    

In [19]:
import joblib
from sklearn.preprocessing import LabelEncoder
import os

# 定义需要编码的分类特征
# categorical_columns = ['flt_no', 'bd_type', 'aircraft', 'a', 'b', 'c', 'from', 'to']
categorical_columns = ['flt_no', 'bd_type', 'aircraft']

# 创建并应用 LabelEncoder
label_encoders = {}

# 创建保存编码器的文件夹（如果文件夹不存在）
save_folder = '../../data-hh/my/encoder/'
os.makedirs(save_folder, exist_ok=True)  # 如果文件夹已存在，不会报错

# 遍历每个分类特征，使用 LabelEncoder 对其进行编码
for col in categorical_columns:
    le = LabelEncoder()  # 创建一个 LabelEncoder 实例
    data[col] = le.fit_transform(data[col])  # 对训练数据中的分类特征进行编码
    label_encoders[col] = le  # 将每个特征的编码器保存到字典中，方便后续使用

    # 保存每个编码器到指定文件夹
    encoder_path = os.path.join(save_folder, f"{col}_encoder_all.pkl")  # 构建保存路径
    joblib.dump(le, encoder_path)  # 使用 joblib 将编码器保存为 pkl 文件
    print(f"{col} 的编码器已保存为 {encoder_path}")  # 输出保存的路径

flt_no 的编码器已保存为 ../../data-hh/my/encoder/flt_no_encoder_all.pkl
bd_type 的编码器已保存为 ../../data-hh/my/encoder/bd_type_encoder_all.pkl
aircraft 的编码器已保存为 ../../data-hh/my/encoder/aircraft_encoder_all.pkl


## 特征和目标分离
我们要预测的是pax字段，其他字段作为特征。

In [21]:
# 特征列
# X = data[['flt_no', 'bd_type', 'cap', 'aircraft',  'leg_no', 'duration', 'a', 'b', 'c', 'year', 'month', 'day', 'weekday','holiday', 'hour', 'minute', 'second', 'from', 'to','unit_price']]
# X = data[['flt_no', 'bd_type', 'cap', 'aircraft', 'legs', 'leg_no', 'duration', 'a', 'b', 'c', 'year', 'month', 'day', 'weekday','hour', 'minute', 'second', 'from', 'to','unit_price']]
X = data[['flt_no', 'bd_type', 'cap', 'aircraft', 'legs', 'leg_no', 'duration', 'a', 'b', 'c', 'year', 'month', 'day', 'weekday','hour', 'minute', 'second', 'from', 'to','unit_price','a_label' ,'b_label' ,'c_label' ,'from_label' ,'to_label']]

# 目标列
y = data['pax']

## 训练XGBoost模型

In [22]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np

# 自定义 SMAPE 函数
def smape(y_true, y_pred):
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    diff = np.abs(y_true - y_pred)
    return np.mean(diff / denominator) * 100

# 自定义评估函数
def smape_eval(y_pred, dtrain):
    y_true = dtrain.get_label()
    smape_value = smape(y_true, y_pred)
    return 'SMAPE', smape_value

# 数据划分
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# 输出数据集大小
print(f'训练集大小: {X_train.shape[0]}')
print(f'验证集大小: {X_val.shape[0]}')
print(f'测试集大小: {X_test.shape[0]}')

# 转换为 DMatrix 格式
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)
dtest = xgb.DMatrix(X_test, label=y_test)

# 设置参数
params = {
    'objective': 'reg:squarederror',
    'learning_rate': 0.01,
    'max_depth': 6,
    'subsample': 0.8,
    'colsample_bytree': 0.7,
    'alpha': 10
}

# 设置评估集
evals = [(dtrain, 'train'), (dval, 'validation')]

# 训练模型，使用自定义评估指标
model = xgb.train(
    params,
    dtrain,
    num_boost_round=1000,
    evals=evals,
    early_stopping_rounds=10,
    custom_metric=smape_eval,  # 使用 custom_metric 参数
    verbose_eval=10 #隔多少轮显示一次
)

# 预测测试集
y_pred = model.predict(dtest)

# 测试集 SMAPE 评估
test_smape = smape(y_test, y_pred)
print(f'SMAPE on Test Set: {test_smape:.2f}%')

# 测试集 MSE 评估
mse = mean_squared_error(y_test, y_pred)
print(f'Mean Squared Error on Test Set: {mse}')


训练集大小: 4799380
验证集大小: 599922
测试集大小: 599923
[0]	train-rmse:45.27172	train-SMAPE:35.69323	validation-rmse:45.29044	validation-SMAPE:35.70506
[10]	train-rmse:42.85397	train-SMAPE:34.21264	validation-rmse:42.87737	validation-SMAPE:34.22837
[20]	train-rmse:40.79329	train-SMAPE:32.92702	validation-rmse:40.82104	validation-SMAPE:32.94536
[30]	train-rmse:38.95741	train-SMAPE:31.74132	validation-rmse:38.98899	validation-SMAPE:31.76209
[40]	train-rmse:37.41744	train-SMAPE:30.70917	validation-rmse:37.45237	validation-SMAPE:30.73146
[50]	train-rmse:36.07731	train-SMAPE:29.78321	validation-rmse:36.11573	validation-SMAPE:29.80730
[60]	train-rmse:34.82429	train-SMAPE:28.88423	validation-rmse:34.86602	validation-SMAPE:28.90931
[70]	train-rmse:33.80455	train-SMAPE:28.12193	validation-rmse:33.84889	validation-SMAPE:28.14763
[80]	train-rmse:32.90492	train-SMAPE:27.41972	validation-rmse:32.95156	validation-SMAPE:27.44643
[90]	train-rmse:32.13635	train-SMAPE:26.79685	validation-rmse:32.18478	validation-SMA

[840]	train-rmse:26.21110	train-SMAPE:21.12077	validation-rmse:26.27633	validation-SMAPE:21.14393
[850]	train-rmse:26.19996	train-SMAPE:21.11273	validation-rmse:26.26525	validation-SMAPE:21.13583
[860]	train-rmse:26.18669	train-SMAPE:21.10269	validation-rmse:26.25213	validation-SMAPE:21.12589
[870]	train-rmse:26.17118	train-SMAPE:21.09077	validation-rmse:26.23682	validation-SMAPE:21.11395
[880]	train-rmse:26.15537	train-SMAPE:21.07922	validation-rmse:26.22113	validation-SMAPE:21.10242
[890]	train-rmse:26.14307	train-SMAPE:21.07062	validation-rmse:26.20904	validation-SMAPE:21.09386
[900]	train-rmse:26.12700	train-SMAPE:21.05863	validation-rmse:26.19302	validation-SMAPE:21.08188
[910]	train-rmse:26.11617	train-SMAPE:21.05097	validation-rmse:26.18244	validation-SMAPE:21.07432
[920]	train-rmse:26.10473	train-SMAPE:21.04280	validation-rmse:26.17107	validation-SMAPE:21.06613
[930]	train-rmse:26.09477	train-SMAPE:21.03557	validation-rmse:26.16119	validation-SMAPE:21.05897
[940]	train-rmse:26.

目前的情况是统计城市出现的频率，效果比使用标签编码好，这个也是有理由的，因为一个城市的航班次数越多，肯定说明需求多，航空公司才安排这么多人，所以说航班次数和客流直接也是存在关联的，所以增强了模型的效果

In [9]:
# 显示20条测试结果（真实值 vs 预测值）
test_results = list(zip(y_test.values[:100], y_pred[:100]))  # 真实值和预测值
print("\n20条测试结果（真实值 vs 预测值）:")
for i, (true_value, pred_value) in enumerate(test_results):
    print(f"第{i+1}条: 真实值={true_value}, 预测值={pred_value:.2f}")


20条测试结果（真实值 vs 预测值）:
第1条: 真实值=91, 预测值=108.46
第2条: 真实值=255, 预测值=214.25
第3条: 真实值=163, 预测值=128.98
第4条: 真实值=234, 预测值=200.29
第5条: 真实值=82, 预测值=120.99
第6条: 真实值=88, 预测值=93.80
第7条: 真实值=66, 预测值=73.23
第8条: 真实值=224, 预测值=196.98
第9条: 真实值=181, 预测值=165.41
第10条: 真实值=131, 预测值=116.64
第11条: 真实值=116, 预测值=152.21
第12条: 真实值=99, 预测值=126.16
第13条: 真实值=127, 预测值=121.73
第14条: 真实值=30, 预测值=71.22
第15条: 真实值=106, 预测值=93.30
第16条: 真实值=155, 预测值=134.01
第17条: 真实值=139, 预测值=157.82
第18条: 真实值=28, 预测值=59.45
第19条: 真实值=88, 预测值=118.55
第20条: 真实值=200, 预测值=176.92
第21条: 真实值=102, 预测值=136.95
第22条: 真实值=93, 预测值=105.33
第23条: 真实值=118, 预测值=122.86
第24条: 真实值=182, 预测值=135.34
第25条: 真实值=69, 预测值=115.13
第26条: 真实值=49, 预测值=64.17
第27条: 真实值=70, 预测值=74.06
第28条: 真实值=146, 预测值=118.50
第29条: 真实值=31, 预测值=44.18
第30条: 真实值=179, 预测值=163.37
第31条: 真实值=168, 预测值=158.15
第32条: 真实值=174, 预测值=189.82
第33条: 真实值=124, 预测值=108.58
第34条: 真实值=96, 预测值=79.65
第35条: 真实值=132, 预测值=113.02
第36条: 真实值=162, 预测值=123.75
第37条: 真实值=144, 预测值=127.16
第38条: 真实值=185, 预测值=181.71
第39条: 真实值=130, 预测值=118

似乎对于较小值预测存在误差

In [10]:
def calculate_smape(y_true, y_pred):
    """
    计算 Symmetric Mean Absolute Percentage Error (SMAPE)
    """
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    smape = 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred)))
    return smape

def calculate_mape(y_true, y_pred):
    """
    计算 Mean Absolute Percentage Error (MAPE)
    """
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mape = 100 * np.mean(np.abs((y_true - y_pred) / y_true))
    return mape

In [11]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# 评估模型
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mse)
mape = calculate_mape(y_test, y_pred)
smape = calculate_smape(y_test, y_pred)

# 打印结果
print(f'Mean Squared Error (MSE): {mse:.4f}')
print(f'Root Mean Squared Error (RMSE): {rmse:.4f}')
print(f'Mean Absolute Error (MAE): {mae:.4f}')
print(f'Mean Absolute Percentage Error (MAPE): {mape:.4f}%')
print(f'Symmetric Mean Absolute Percentage Error (SMAPE): {smape:.4f}%')

Mean Squared Error (MSE): 676.4863
Root Mean Squared Error (RMSE): 26.0094
Mean Absolute Error (MAE): 20.4320
Mean Absolute Percentage Error (MAPE): 25.1024%
Symmetric Mean Absolute Percentage Error (SMAPE): 20.9867%


## 保存模型

In [ ]:
model.save_model("../../data-hh/my/模型文件/频率编码/xgboost_model_1000.json")
print("模型已保存为 xgboost_model.json")

## 超参数设置

## 不同特征重要程度测试

In [ ]:
import xgboost as xgb
import matplotlib.pyplot as plt

# 假设 model 是训练好的 XGBoost 模型
xgb.plot_importance(model, importance_type='weight', title="Feature Importance (Weight)", height=0.5)
plt.show()

xgb.plot_importance(model, importance_type='gain', title="Feature Importance (Gain)", height=0.5)
plt.show()

xgb.plot_importance(model, importance_type='cover', title="Feature Importance (Cover)", height=0.5)
plt.show()